In [28]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

In [29]:
df_train = pd.read_csv("../data/processed/train_pymes_2017_2022.csv")
df_test = pd.read_csv("../data/processed/test_pymes_2023_2025.csv")

# Variables X y Y
X_train = df_train.drop(columns=["target"]) 
y_train = df_train["target"]

X_test = df_test.drop(columns=["target"]) 
y_test = df_test["target"]

In [30]:
# Modelo de regresion logistica
baseline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
])

In [31]:
# Modelo de Random Forest
challenger_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

In [32]:
# Funcion de evaluacion
def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te):
    # Entrenamiento
    modelo.fit(X_tr, y_tr)

    # Inferencia
    y_pred = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]
    
    roc_auc = roc_auc_score(y_te, y_proba)
    pr_auc = average_precision_score(y_te, y_proba)
    
    print(f"================== {nombre} ==================")
    print(f"ROC-AUC (2023-2025): {roc_auc:.4f}")
    print(f"PR-AUC (Average Precision): {pr_auc:.4f}\n")
    print("Reporte de Clasificación:")
    print(classification_report(y_te, y_pred, target_names=["Saludable (0)", "Estrés (1)"]))
    print("Matriz de Confusión:")
    print(confusion_matrix(y_te, y_pred))
    print("\n")
    
    return {
        "modelo": nombre,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "estimator": modelo
    }

In [33]:
# Experimento comparativo
res_lr = evaluar_modelo("Baseline: Regresión Logística", baseline_lr, X_train, y_train, X_test, y_test)
res_rf = evaluar_modelo("Challenger: Random Forest", challenger_rf, X_train, y_train, X_test, y_test)

================== Baseline: Regresión Logística ==================
ROC-AUC (2023-2025): 0.8820
PR-AUC (Average Precision): 0.7352

Reporte de Clasificación:
               precision    recall  f1-score   support

Saludable (0)       0.94      0.91      0.92     57249
   Estrés (1)       0.65      0.74      0.69     13541

     accuracy                           0.87     70790
    macro avg       0.79      0.82      0.81     70790
 weighted avg       0.88      0.87      0.88     70790

Matriz de Confusión:
[[51967  5282]
 [ 3583  9958]]


================== Challenger: Random Forest ==================
ROC-AUC (2023-2025): 0.9369
PR-AUC (Average Precision): 0.8269

Reporte de Clasificación:
               precision    recall  f1-score   support

Saludable (0)       0.95      0.93      0.94     57249
   Estrés (1)       0.74      0.80      0.77     13541

     accuracy                           0.91     70790
    macro avg       0.85      0.87      0.86     70790
 weighted avg       0.91